# 00. От промпта к видео: карта и ресурсы

Цель: понять роли всех компонентов до загрузки весов. Итоговый ориентир — T2V Lite distilled16steps 5s; учебный режим работает на M3 Max и в CPU Colab.


### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


### 1. Модель — это система

Представьте фильм как маршрут: текст задаёт пункт назначения, VAE определяет язык компактных кадров, DiT подсказывает движение, sampler делает шаги. Токенизатор преобразует строку в номера; текстовый энкодер преобразует номера в векторы. Они решают разные задачи.


In [ ]:
from labkit.components import REPOS
for component,repo in REPOS.items(): print(component, '→', repo)


### 2. Читаем настоящий рецепт

YAML не содержит веса: это размеры архитектуры и настройки запуска. Проверьте 16 шагов, guidance=1, 32 visual blocks, 2 text blocks. `attention.type=flash` обозначает dense ветку генератора; фактический движок может быть SDPA.


In [ ]:
from omegaconf import OmegaConf
conf=OmegaConf.load(SOURCE/'configs/k5_lite_t2v_5s_distil_sd.yaml')
print(OmegaConf.to_yaml(conf))


### 3. Параметры и память

BF16 хранит один параметр в 2 байтах, FP32 в 4. Это только веса: нужны ещё активации, временные копии и память ОС. На Mac CPU и GPU используют общий пул; offload не создаёт дополнительную физическую RAM. Qwen загружается целиком, включая неиспользуемые в T2V vision/LM части.


In [ ]:
for label,count in [('Video DiT (округлено)',2e9),('Qwen (округлено)',7e9),('Image DiT (округлено)',6e9)]:
    print(label, 'BF16 GiB=', round(count*2/2**30,2), 'FP32 GiB=',round(count*4/2**30,2))


### 4. Размер латентного видео

VAE сжимает пространство в 8 раз и время в 4. Для официального 5s пути получается 31 латентный кадр. Декодер даёт 121 кадр: 4×(31−1)+1. При 24 fps это около 5.04 s; название 5s — условное. Patch 1×2×2 даёт 47 616 visual tokens при 512×768.


In [ ]:
seconds,H,W=5,512,768
T=seconds*24//4+1
N=T*(H//16)*(W//16)
print('latent:',(1,16,T,H//8,W//8),'tokens:',N,'decoded frames:',4*(T-1)+1)
print('Наивная BF16 матрица attention для 28 heads, GiB:',round(28*N*N*2/2**30,1))


### 5. Frozen и trainable — проверяем на маленькой сети

Заморозить означает отключить градиенты весов. `eval()` меняет режим Dropout/BatchNorm, но само по себе не отключает обучение. `no_grad()` отключает запись графа данного вычисления. При обучении генератора внешние энкодеры можно заморозить, а адаптер внутри DiT продолжит учиться.


In [ ]:
external=torch.nn.Linear(4,8).requires_grad_(False).eval()
trainable=torch.nn.Linear(8,2)
x=torch.randn(3,4)
loss=trainable(external(x)).square().mean(); loss.backward()
print('external gradient:',external.weight.grad)
print('DiT adapter gradient norm:',trainable.weight.grad.norm().item())


### 6. Где заканчиваются подтверждённые факты

README курса содержит training map с источниками. Код инференса явно замораживает Qwen/CLIP; VAE загружается из стороннего checkpoint. Полная историческая настройка optimizer всех этапов не опубликована в этом репозитории. Наш учебный training loop учит маленький DiT; он не воспроизводит промышленное обучение.


In [ ]:
source_excerpt('kandinsky/models/text_embedders.py','self.model = freeze',8)
source_excerpt('download_models.py','Qwen/Qwen2.5',9)


#


In [ ]:
#


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
